# PepXPro Tutorial

**PepXPro** is a modular framework for building peptide–protein benchmark datasets from PDBbind.  
It downloads, fixes, annotates, and samples co-crystal structures with experimental binding affinities.

**Pipeline overview:**
```
PDBbind INDEX
     │
     ▼
scrape/                                       (pepxpro scrape -v runs the four standard steps)
  pepxpro scrape --steps download        →  pdb_cache/  +  peptide_entries.tsv
  pepxpro scrape --steps filter          →  fixed_cif/  +  filtered_systems.tsv
  pepxpro scrape --steps summary         →  summary.tsv
  pepxpro scrape --external-input FILE   →  summary_merged.tsv  (optional: add external datasets)
  pepxpro scrape --steps distributions   →  plots/
     │
     ▼
genSample/
  pepxpro genSample gen       →  run_XX/sample.tsv  +  structures  +  FASTA
  pepxpro genSample methods   →  sampling comparison plot
  pepxpro genSample plot      →  per-run distribution plot
  pepxpro genSample view      →  PyMOL interactive viewer
```

Every `pepxpro <subcommand>` above also works as `python pepxpro.py <subcommand> ...`, or by
running the underlying script directly (e.g. `python genSample/scripts/01_gen_sample.py ...`) —
see the README and `-h`/`--help` on any command for the full flag reference.

**Contents**
1. Setup
2. scrape — building the dataset
3. External datasets — integrating PEPBI
4. genSample — filtering, sampling, export
5. PyMOL viewer
6. Applications
   - A. MD simulation input (CHARMM-GUI / GROMACS)
   - B. Docking benchmark
   - C. ML ΔG prediction dataset
   - D. AlphaFold-Multimer benchmark
   - E. Peptide design / lead optimization
   - F. Virtual screening validation


---
## 1. Setup

In [ ]:
import os
import shutil
from pathlib import Path
from IPython.display import Image, display

# Repository root — adjust if running from a different location
REPO = Path("../").resolve()
OUTPUTS = REPO / "scrape" / "outputs"

print("Repository root:", REPO)
print("Conda env:", os.environ.get("CONDA_DEFAULT_ENV", "(not detected — run: conda activate pepxpro)"))
print("pepxpro command:", shutil.which("pepxpro") or "NOT FOUND — run: python -m pip install -e . from the repo root")


---
## 2. scrape

Runs the four-step pipeline to download and prepare all peptide–protein systems from PDBbind.

> **Prerequisite:** place `INDEX_general_PL.2020R1.lst` (the refined v2020R1 index — PDBbind
> registration required) in `scrape/inputs/`. This is the default `01_download.py` reads;
> pass `--index scrape/inputs/INDEX_general_PL.2020` (from the repo root) to use the original v2020 index instead.
> Only this single index file is needed — PDBbind data is not distributed with PepXPro (see README, *PDBbind data and licensing*).


### Step 1 — Download
Parses the index for n-mer entries with Kd/Ki affinity.  
Downloads biological assembly CIFs from RCSB (~750 entries, ~30 min).

In [ ]:
!pepxpro scrape -v --steps download


### Step 2 — Filter and fix
Selects best assembly, identifies chains, checks AMBER14 compatibility,  
fills missing residues with PDBFixer, runs DSSP, annotates receptor functional class.  
**Incremental:** re-running skips cached entries; only new ones are processed.

In [ ]:
!pepxpro scrape -v --steps filter


### Step 3 — Summary
Joins filter results with affinity data; computes SS fractions; sorts by ΔG.

In [ ]:
!pepxpro scrape -v --steps summary


In [ ]:
import pandas as pd
df = pd.read_csv(OUTPUTS / "summary.tsv", sep="\t")
print(f"{len(df)} entries  |  columns: {list(df.columns)}")
df[["pdb_id","dG_kcal_mol","binding_type","resolution","year",
    "receptor_func_class","peptide_pct_helix"]].head(10)

### Step 4 — Distributions
Box plots for all numeric columns in summary.tsv.

In [ ]:
!pepxpro scrape -v --steps distributions


In [ ]:
display(Image(str(OUTPUTS / "plots" / "summary_distributions.png"), width=2000))

---
## 3. External datasets

PepXPro can integrate any external affinity dataset (xlsx / tsv / csv) alongside PDBbind.  
This example uses **PEPBI** — 329 ITC-measured peptide–protein affinities (Dryad, doi:10.5061/dryad.wstqjq2wk).

What the script does:
1. Auto-detects column mapping (saves `pepbi_v1_colmap.json` for future runs)
2. Deduplicates against PDBbind
3. Downloads missing CIFs; fills year + resolution from CIF and RCSB API
4. Runs filter + summary on the new entries
5. Merges with `summary.tsv` → `summary_merged.tsv`

In [ ]:
!pepxpro scrape -v --steps external \
    --external-input {REPO}/scrape/inputs/external/PEPBI.xlsx \
    --name pepbi_v1


In [ ]:
merged = pd.read_csv(OUTPUTS / "summary_merged.tsv", sep="\t")
print(f"Merged dataset: {len(merged)} entries")
merged["ext_source"].fillna("PDBbind").value_counts()

To add a second external dataset later and rebuild the merged file:
```bash
pepxpro scrape -v --steps external --external-input scrape/inputs/external/my_dataset.tsv --name custom_v1
```
`--merge-only`, `--no-download`, etc. are extra `04_external.py` flags not exposed by the
`pepxpro` wrapper — for those, run the script directly:
```bash
cd scrape/scripts
python 04_external.py --merge-only
```


---
## 4. genSample

Flexible pipeline to filter, deduplicate, sample, and export datasets.  
Each run writes to a new numbered directory (`run_01`, `run_02`, …) — nothing is overwritten.  
`run_config.json` records every parameter for full reproducibility.

### Dry run — inspect the pool before sampling

In [ ]:
!pepxpro genSample gen \
    --dry-run \
    --max-peptide 15 \
    --min-year 2005 \
    --binding-type kd \
    --summary-tsv {REPO}/scrape/outputs/summary_merged.tsv


### Basic stratified sample — 20 entries

In [ ]:
_out = !pepxpro genSample gen --sample 20 --summary-tsv {REPO}/scrape/outputs/summary_merged.tsv --seed 42
print("\n".join(_out))
import re
demo_run = re.search(r"Output → (\S+)", "\n".join(_out)).group(1).rsplit("/", 1)[-1]
print(f"\n[captured] demo_run = {demo_run}")


### Quality + redundancy filters

In [ ]:
!pepxpro genSample gen \
    --sample 30 \
    --single-chain \
    --max-peptide 15 \
    --max-resolution 2.5 \
    --min-year 2005 \
    --seq-id-receptor medium \
    --seq-id-peptide strict \
    --summary-tsv {REPO}/scrape/outputs/summary_merged.tsv \
    --seed 42


### Sampling methods

Five algorithms are available. The `--dims` flag selects the feature space.

| Method | Best for |
|---|---|
| `stratified` | uniform grid coverage (default) |
| `kennard-stone` | maximizing pairwise distance in N-D space |
| `kde` | over-representing tails of the affinity distribution |
| `cluster` | one representative per k-means centroid |
| `isolation-forest` | favoring structural/affinity outliers |

In [ ]:
# Compare all five methods on the same pool
!pepxpro genSample methods --n 20 --seed 42 --dims dg pep_len


In [ ]:
display(Image(str(REPO / "genSample" / "outputs" / "plots" / "sampling_benchmark.png"), width=2000))

In [ ]:
# Sample by receptor functional class + affinity
!pepxpro genSample gen \
    --sample 20 \
    --sampling kennard-stone \
    --dims func_class dg \
    --summary-tsv {REPO}/scrape/outputs/summary_merged.tsv \
    --seed 42


### Export — structures and sequences

In [ ]:
# CIF symlinks (fast, no disk copy)
!pepxpro genSample gen --sample 20 --link-cifs --seed 42


In [ ]:
# PDB format, protonated at pH 7.4
!pepxpro genSample gen \
    --sample 20 \
    --copy-cifs --format pdb --protonate --ph 7.4 \
    --seed 42


In [ ]:
# Full metadata + FASTA sequences
!pepxpro genSample gen \
    --sample 20 \
    --full-metadata \
    --export-fasta \
    --summary-tsv {REPO}/scrape/outputs/summary_merged.tsv \
    --seed 42


### Per-run distribution plot

In [ ]:
!pepxpro genSample plot --run {demo_run}


In [ ]:
display(Image(str(REPO / "genSample" / "outputs" / demo_run / "plots" / "run_comparison.png"), width=2000))


---
## 5. PyMOL viewer

Browse the structures of a sample run interactively.  
Structures are displayed in ΔG order (tightest binder first).  
**Arrow keys** to navigate. Chains colored with `util.cbc`.

> This cell launches PyMOL as a separate process — run it from the terminal.

In [ ]:
# First generate a run with CIF copies
_out = !pepxpro genSample gen --sample 20 --copy-cifs --seed 42
print("\n".join(_out))
import re
pymol_run = re.search(r"Output → (\S+)", "\n".join(_out)).group(1).rsplit("/", 1)[-1]
pymol_run_n = int(pymol_run.split("_")[1])
print(f"\n[captured] pymol_run = {pymol_run}  (n={pymol_run_n})")


In [ ]:
# Open PyMOL viewer (launches a separate window)
!pepxpro genSample view --run {pymol_run_n} --cif-dir cif


---
## 6. Applications

---
### A. MD simulation input — CHARMM-GUI / GROMACS

**Goal:** get a single non-protonated/protonated PDB file for a specific complex,  
ready to upload to [CHARMM-GUI Solution Builder](https://charmm-gui.org) or use with GROMACS.

`--pick` bypasses all filters and returns the fixed, protonated structure for any PDB ID.

In [ ]:
!pepxpro genSample gen \
    --pick 2las \
    --summary-tsv {REPO}/scrape/outputs/summary_merged.tsv \
    --format pdb \
    --protonate \
    --ph 7.4


The output PDB has:
- Missing residues filled by PDBFixer
- AMBER14-compatible residue names
- Hydrogens added at the specified pH (OPTIONAL)
- Peptide and receptor as separate chains

```bash
# CHARMM-GUI: upload run_XX/pdb/2las.pdb → Solution Builder

# GROMACS:
gmx pdb2gmx -f 2las.pdb -ff amber14sb -water tip3p
```

To prepare a set of related complexes at once:

In [ ]:
!pepxpro genSample gen \
    --no-sample \
    --max-receptor 200 \
    --min-receptor 180 \
    --seq-id-receptor strict \
    --format pdb \
    --protonate \
    --ph 7.4


---
### B. Docking benchmark dataset

**Goal:** a diverse, non-redundant set of peptide–protein complexes with known binding poses  
and experimental affinities, for validating docking protocols.

**Requirements:** receptor fold diversity, peptide length range, high-quality crystal structures.

> **Requires `cd-hit`** for receptor sequence deduplication (`--seq-id-receptor`).  
> Install once (if not already present): `conda install -c conda-forge -c bioconda cd-hit`

In [ ]:
!pepxpro genSample gen \
    --sample 20 \
    --single-chain \
    --max-resolution 2.5 \
    --max-peptide 15 \
    --seq-id-receptor strict \
    --tm-score-receptor 0.5 \
    --sampling kennard-stone \
    --dims dg pep_len rec_len \
    --copy-cifs --format pdb \
    --full-metadata \
    --seed 42


**Evaluation workflow:**
1. Strip peptide chain from each PDB → apo receptor for blind re-docking
2. Run your docking protocol (AutoDock-GPU, Glide, HADDOCK, …)
3. Evaluate pose RMSD against the co-crystal peptide chain
4. Use `dG_kcal_mol` from `sample.tsv` for enrichment metrics (ROC-AUC, EF1%)

The Kennard-Stone sampler in 3D (ΔG × peptide length × receptor length) ensures  
the benchmark spans the full complexity space rather than the most populated region.

---
### C. ML ΔG prediction dataset

**Goal:** a clean, split-ready dataset for training or benchmarking binding affinity models  
(sequence-based, structure-based, or graph neural networks).

In [ ]:
!pepxpro genSample gen \
    --no-sample \
    --seq-id-receptor medium \
    --seq-id-peptide medium \
    --max-added 3 \
    --max-receptor-added 5 \
    --full-metadata \
    --export-fasta \
    --summary-tsv {REPO}/scrape/outputs/summary_merged.tsv


In [ ]:
import glob
latest_run = sorted(glob.glob(str(REPO / "genSample" / "outputs" / "run_*")))[-1]
sample = pd.read_csv(f"{latest_run}/sample.tsv", sep="\t")
print(f"Dataset size: {len(sample)} entries")
print(f"\nΔG range: {sample.dG_kcal_mol.min():.2f} to {sample.dG_kcal_mol.max():.2f} kcal/mol")
print(f"\nReceptor functional classes:")
print(sample.receptor_func_class.value_counts())

**Key columns for ML:**

| Column | Use |
|---|---|
| `dG_kcal_mol` | Regression target |
| `peptide_seq` | Peptide sequence features |
| `receptor_seq` | Receptor sequence features |
| `peptide_pct_helix/beta/coil` | SS fraction features |
| `receptor_func_class` | Receptor category (enzyme, kinase, …) |
| `year` | For temporal train/test splits |
| `resolution` | Structure quality filter/feature |

**Suggested train/test splits:**
- **Temporal:** `year <= 2018` → train, `year > 2018` → test
- **Sequence:** CD-HIT at 30% on receptor_seq to define splits
- **Random:** 80/10/10 by row

---
### D. AlphaFold-Multimer benchmark

**Goal:** experimental peptide–protein complexes to evaluate co-folding models  
(AlphaFold-Multimer, RoseTTAFold2-NA, Chai-1, …) on binding mode prediction.

**Selection criteria:** single-chain receptor, short peptides (≤ 20 aa), high-resolution crystal, diverse receptor folds.

In [ ]:
!pepxpro genSample gen \
    --sample 20 \
    --single-chain \
    --max-peptide 20 \
    --max-resolution 2.0 \
    --seq-id-receptor strict \
    --seq-id-peptide medium \
    --sampling kennard-stone \
    --dims dg pep_len \
    --export-fasta \
    --copy-cifs \
    --full-metadata \
    --seed 42


**Evaluation workflow:**
1. Feed `(receptor_seq, peptide_seq)` pairs from the FASTA files to AF-Multimer
2. Align predicted complex to the experimental CIF (receptor backbone superposition)
3. Compute peptide Cα RMSD at the interface
4. Correlate pTM / ipTM scores with actual interface RMSD

The 30% receptor sequence identity cutoff ensures diverse folds —  
not just well-sampled families like kinases and PDZ domains.

---
### E. Peptide design / lead optimization

**Goal:** retrieve all known binders for a target receptor as structural and sequence templates  
for peptide design or in silico optimization.

**Example:** designing a p53-mimetic peptide inhibitor of MDM2.

In [ ]:
# Retrieve the best known MDM2-peptide structure in the dataset (3lnz: 12-mer, dG = -12.78 kcal/mol)
!pepxpro genSample gen \
    --pick 3lnz \
    --format pdb \
    --protonate \
    --ph 7.0


In [ ]:
# Get all complexes in the MDM2 receptor size range (~140-220 aa)
# with full metadata and sequences for alignment
!pepxpro genSample gen \
    --no-sample \
    --min-receptor 140 --max-receptor 220 \
    --max-peptide 20 \
    --full-metadata \
    --export-fasta \
    --copy-cifs --format pdb


**Downstream workflow:**
```bash
# MSA of known binders
muscle -in run_XX/peptide_sequences.fasta -out binders_aligned.fasta

# Sequence design with ProteinMPNN
python run_ProteinMPNN.py --pdb_path run_XX/pdb/3lnz.pdb ...

# Validate designed sequences with AF-Multimer before synthesis
```

The `dG_kcal_mol` column provides the affinity ladder of known binders —  
your design target is to improve on the best natural peptide.

---
### F. Virtual screening validation set

**Goal:** a balanced set of strong binders (positives) and weak binders (negatives)  
for computing ROC curves and validating a scoring function or VS pipeline.

**Strategy:** split by ΔG threshold with a gap to avoid ambiguous mid-affinity cases.

In [ ]:
# Positives — strong binders (dG ≤ -9 kcal/mol)
!pepxpro genSample gen \
    --sample 20 \
    --max-dg -9.0 \
    --single-chain \
    --seq-id-receptor medium \
    --sampling kennard-stone \
    --dims dg pep_len \
    --copy-cifs \
    --full-metadata \
    --seed 42


In [ ]:
# Negatives — weak binders (dG ≥ -6 kcal/mol)
!pepxpro genSample gen \
    --sample 20 \
    --min-dg -6.0 \
    --single-chain \
    --seq-id-receptor medium \
    --sampling kennard-stone \
    --dims dg pep_len \
    --copy-cifs \
    --full-metadata \
    --seed 42


In [ ]:
# Combine and label
import glob
runs = sorted(glob.glob(str(REPO / "genSample" / "outputs" / "run_*")))
pos = pd.read_csv(f"{runs[-2]}/sample.tsv", sep="\t"); pos["active"] = 1
neg = pd.read_csv(f"{runs[-1]}/sample.tsv", sep="\t"); neg["active"] = 0
vs_set = pd.concat([pos, neg]).reset_index(drop=True)
print(f"VS set: {len(vs_set)} complexes  |  positives: {len(pos)}  negatives: {len(neg)}")
vs_set[["pdb_id","dG_kcal_mol","active"]].head()

**Evaluation:**
```python
from sklearn.metrics import roc_auc_score

# vs_set["score"] = your scoring function output
roc_auc_score(vs_set["active"], vs_set["score"])
```

---
## Tips

**Reproducing a run:**
```bash
cat genSample/outputs/run_01/run_config.json
```

**Checking pool size before committing:**  
Always `--dry-run` first when exploring new filter combinations.

**Disk usage:**  
`--link-cifs` uses symlinks (negligible disk). Use `--copy-cifs` only for portable run directories.

**Updating after a new PDBbind release:**  
Re-run `pepxpro scrape -v --steps download` then `--steps filter`. The cache mechanism skips existing entries.
